In [1]:
# Split integrity + directory-name semantics for the grit-entity-new subset.
# Colab cell: mounts Drive, reads train/val/test.jsonl directly from Drive. No local copies.

import json, re, collections, os
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"

def load(s):
    with open(os.path.join(DATA, f"{s}.jsonl"), encoding="utf-8") as f:
        return [json.loads(l) for l in f]

SPLITS = {s: load(s) for s in ["train", "val", "test"]}
ALL    = [r for rows in SPLITS.values() for r in rows]

top  = lambda p: p.split("/")[0]
nref = lambda r: len(r["input_images"])
matK = lambda p: int(re.search(r"-mat-(\d+)-", p.split("/")[-1]).group(1))

# leakage: no target scene and no reference-crop file may cross splits
tgt = {s: {r["output_image"] for r in rows}            for s, rows in SPLITS.items()}
crp = {s: {p for r in rows for p in r["input_images"]} for s, rows in SPLITS.items()}
print("target overlap      train/test %d  val/test %d  train/val %d" % (
      len(tgt["train"] & tgt["test"]), len(tgt["val"] & tgt["test"]), len(tgt["train"] & tgt["val"])))
print("crop-file overlap   train/test %d  val/test %d  train/val %d" % (
      len(crp["train"] & crp["test"]), len(crp["val"] & crp["test"]), len(crp["train"] & crp["val"])))

# reference count per row, by directory (axis = len(input_images), not the dir name)
ref_by_dir = collections.defaultdict(collections.Counter)
for r in ALL:
    ref_by_dir[top(r["output_image"])][nref(r)] += 1
print("\nreference count (len input_images) by directory:")
for d in ["single", "double", "triple"]:
    print("  %-6s %s" % (d, dict(sorted(ref_by_dir[d].items()))))

# distinct entities per target, by directory (mat-K roster; only triple maps 1:1 to its name)
ents, tdir = collections.defaultdict(set), {}
for r in ALL:
    t = r["output_image"]; tdir[t] = top(t)
    ents[t] |= {matK(p) for p in r["input_images"]}
ent_by_dir = collections.defaultdict(collections.Counter)
for t, e in ents.items():
    ent_by_dir[tdir[t]][len(e)] += 1
print("\ndistinct entities per target by directory:")
for d in ["single", "double", "triple"]:
    print("  %-6s %s" % (d, dict(sorted(ent_by_dir[d].items()))))

# references within one row are always distinct entities
dup = sum(1 for r in ALL if len({matK(p) for p in r["input_images"]}) != nref(r))
print("\nrows repeating an entity within one row:", dup)

Mounted at /content/drive
target overlap      train/test 0  val/test 0  train/val 0
crop-file overlap   train/test 0  val/test 0  train/val 0

reference count (len input_images) by directory:
  single {1: 13383, 2: 10987, 3: 2989}
  double {1: 3085, 2: 1509}
  triple {1: 3654, 2: 3654, 3: 1218}

distinct entities per target by directory:
  single {1: 372, 2: 2026, 3: 2985, 4: 1}
  double {1: 68, 2: 1509}
  triple {3: 1218}

rows repeating an entity within one row: 0


In [2]:
from math import comb
import glob

# ref_kind is derived from the "_up" filename flag in score_runs and from the reference
# resolution recorded in input_img_sizes. The two give the same partition on the frozen splits.

assert all(len(r["input_images"]) == len(r["input_img_sizes"]) for r in ALL), \
    "input_images and input_img_sizes differ in length"
pairs = [(("_up" in p), tuple(s) == (800, 800))
         for r in ALL for p, s in zip(r["input_images"], r["input_img_sizes"])]
disagree = sum(1 for a, b in pairs if a != b)
print("reference resolutions:",
      dict(collections.Counter(tuple(s) for r in ALL for s in r["input_img_sizes"])))
print(f"references {len(pairs)} | '_up' vs 800x800 disagreements {disagree}")
assert not disagree, "ref_kind: filename flag and 800x800 resolution define different partitions"

ref_kind = lambda r: ("all_up" if all("_up" in p for p in r["input_images"])
                      else "no_up" if not any("_up" in p for p in r["input_images"]) else "mixed")
for s, rows in SPLITS.items():
    print(f"  {s:5s} {dict(collections.Counter(ref_kind(r) for r in rows))}")

# Rows enumerate the non-empty subsets of a target's entity roster, capped at 3 references per
# row, so a complete target carries sum(C(n,k) for k=1..min(3,n)) rows. The domain filter is per
# row, so a target loses any row whose instruction omits every keyword. Undetected case: a lost
# singleton row whose entity appears in no surviving row.

by_t = collections.defaultdict(list)
for r in ALL:
    by_t[r["output_image"]].append(r)
print("rows per target:", dict(sorted(collections.Counter(len(v) for v in by_t.values()).items())))

expected = lambda n: sum(comb(n, k) for k in range(1, min(3, n) + 1))
short = [(t, n, expected(n), len(v)) for t, v in by_t.items()
         for n in [len({matK(p) for r in v for p in r["input_images"]})] if len(v) != expected(n)]
print(f"targets whose row count differs from the subset enumeration: {len(short)}")
for t, n, exp, got in sorted(short):
    print(f"  {t}: {n} entities -> expect {exp}, have {got}  "
          f"(split: {[s for s in tgt if t in tgt[s]][0]})")

# fewshot_exemplars.jsonl: 5 domain + 5 mismatched reference-free scene descriptions per test row.

uid = lambda r: r["output_image"] + "|" + "|".join(sorted(r["input_images"]))
ex = [json.loads(l) for l in open(os.path.join(DATA, "fewshot_exemplars.jsonl"), encoding="utf-8")]
ex_uids, test_uids = {e["uid"] for e in ex}, {uid(r) for r in SPLITS["test"]}
print(f"exemplar rows {len(ex)} | unique uids {len(ex_uids)} | test uids {len(test_uids)}")
assert len(ex) == len(ex_uids), "duplicate uid in fewshot_exemplars.jsonl"
assert ex_uids == test_uids, "exemplar uids are not exactly the test split"
for key in ("domain", "mismatched"):
    sizes = collections.Counter(len(e[key]) for e in ex)
    tags = sum(1 for e in ex for s in e[key] if "<img>" in s)
    print(f"  {key:11s} set sizes {dict(sizes)} | exemplars containing '<img>' {tags}")
    assert set(sizes) == {5}, f"{key}: not every row carries 5 exemplars"
    assert not tags, f"{key}: a reference tag survived stripping"

# build_fewshot_exemplars draws the mismatched pool from grit-entity-new.jsonl, not from the
# splits. A held-out target enters the pool only through one of its own rows whose instruction
# omits every domain keyword.

KW = ["desktop computer", "computer setup", "laptop", "tablet", "smartphone", "mobile phone",
      "cell phone", "headphones", "earphones", "earbuds", "keyboard", "computer mouse",
      "game controller", "gaming controller", "gaming console", "vr headset",
      "virtual reality headset"]
pat = re.compile("|".join(re.escape(k) for k in KW), re.I)
GRIT = os.path.join(DATA, "grit-entity-new.jsonl")

pool, n_rows, n_lines = set(), 0, 0
with open(GRIT, encoding="utf-8") as f:
    for n_lines, line in enumerate(f):
        r = json.loads(line)
        if r["output_image"].split("/")[0] in ("character", "human"):
            continue
        if not pat.search(r["instruction"]):
            pool.add(r["output_image"]); n_rows += 1
        if n_rows >= 20000:
            break
print(f"mismatched pool: {len(pool)} targets / {n_rows} rows (break at file line {n_lines})")
for s in ("train", "val", "test"):
    print(f"  overlap with {s:5s} targets: {len(pool & tgt[s])}")
assert not (pool & (tgt["test"] | tgt["val"])), "mismatched pool contains a held-out target"

# Score rows carry uid and i; i is a 0-based index into the frozen split. No score value is read.
# baseline_v1 and sd_reps carry no uid field and are skipped.

idx = {s: {i: uid(r) for i, r in enumerate(load(s))}
       for s in ("test", "val", "screen", "screen_small")
       if os.path.exists(os.path.join(DATA, f"{s}.jsonl"))}
checked, skipped = 0, []
for path in sorted(glob.glob(os.path.join(DATA, "scores", "*.jsonl"))):
    name = os.path.basename(path)[:-6]
    split = name.split("__")[0] if "__" in name else "test"
    rows = [r for r in (json.loads(l) for l in open(path, encoding="utf-8")) if "error" not in r]
    if not rows or "uid" not in rows[0] or split not in idx:
        skipped.append(name); continue
    bad = [r["i"] for r in rows if idx[split].get(r["i"]) != r["uid"]]
    dup = len(rows) - len({r["i"] for r in rows})
    assert not bad, f"{name}: uid/i mismatch on rows {bad[:5]}"
    assert not dup, f"{name}: {dup} duplicate i values"
    checked += 1
print(f"score files verified against the frozen splits: {checked}")
print(f"skipped (no uid field): {', '.join(skipped)}")

# Re-derives the domain subset from grit-entity-new.jsonl and checks it against the frozen splits.
# Full 676,603-row pass. Off by default.

RE_DERIVE_CORPUS = False

if RE_DERIVE_CORPUS:
    KEEP = {"single", "double", "triple"}
    groups, refs, excl = collections.defaultdict(int), collections.Counter(), collections.Counter()
    both = collections.defaultdict(lambda: [0, 0])
    with open(GRIT, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            hit = bool(pat.search(r["instruction"]))
            if r["output_image"].split("/")[0] not in ("character", "human"):
                both[r["output_image"]][0 if hit else 1] += 1
            if not hit:
                continue
            dirs = {p.split("/")[0] for p in r["input_images"] + [r["output_image"]] if p}
            if not dirs or not dirs.issubset(KEEP):
                for d in dirs:
                    excl[d] += 1
                continue
            groups[r["output_image"]] += 1
            refs[len(r["input_images"])] += 1
    n = sum(groups.values())
    print(f"domain rows {n} | targets {len(groups)} | ref spread {dict(sorted(refs.items()))}")
    print(f"keyword hits outside single/double/triple: {dict(excl)}")
    assert n == sum(len(v) for v in SPLITS.values()), "domain rows != frozen split total"
    assert set(groups) == tgt["train"] | tgt["val"] | tgt["test"], "corpus targets != union of splits"
    orphan = {t: v for t, v in both.items() if v[0] and v[1]}
    print(f"targets with both keyword and non-keyword rows "
          f"(the only mismatched-pool leak route): {len(orphan)}")
    for t, v in orphan.items():
        print(f"  {t}: {v[0]} kept, {v[1]} dropped, held out: {t in tgt['test'] | tgt['val']}")
else:
    print("RE_DERIVE_CORPUS=False - skipped")

reference resolutions: {(800, 800): 38036, (960, 1280): 23767, (768, 768): 3240}
references 65043 | '_up' vs 800x800 disagreements 0
  train {'all_up': 17041, 'no_up': 12432, 'mixed': 10420}
  val   {'all_up': 39, 'no_up': 37, 'mixed': 22}
  test  {'all_up': 198, 'no_up': 157, 'mixed': 133}
rows per target: {1: 440, 2: 1, 3: 3534, 7: 4203, 14: 1}
targets whose row count differs from the subset enumeration: 1
  double/target/19_2387.png: 2 entities -> expect 3, have 2  (split: train)
exemplar rows 488 | unique uids 488 | test uids 488
  domain      set sizes {5: 488} | exemplars containing '<img>' 0
  mismatched  set sizes {5: 488} | exemplars containing '<img>' 0
mismatched pool: 5583 targets / 20000 rows (break at file line 38461)
  overlap with train targets: 0
  overlap with val   targets: 0
  overlap with test  targets: 0
score files verified against the frozen splits: 68
skipped (no uid field): baseline_v1, omnicontext__baseline, omnicontext__baseline_shipped, omnicontext__lora_r8

In [3]:
# screen carve + training exclusion. screen.jsonl is drawn from train by target; the 62
# targets in screen_exclude_from_training.txt are what every training notebook removes.

sc   = load("screen")
sc_t = {r["output_image"] for r in sc}
ex   = {l.strip() for l in open(os.path.join(DATA, "screen_exclude_from_training.txt"), encoding="utf-8") if l.strip()}

assert (len(SPLITS["train"]), len(SPLITS["val"]), len(SPLITS["test"]), len(sc)) == (39893, 98, 488, 302), "split row counts moved"
assert (len(tgt["train"]), len(tgt["val"]), len(tgt["test"]), len(sc_t)) == (8059, 20, 100, 62), "split target counts moved"
assert sc_t <= tgt["train"], "screen target outside train"
assert ex == sc_t, "exclude file != screen targets"
kept = sum(r["output_image"] not in ex for r in SPLITS["train"])
assert kept == 39591, f"post-exclusion training pool {kept} != 39591"
for s, rows in list(SPLITS.items()) + [("screen", sc)]:
    assert len({uid(r) for r in rows}) == len(rows), f"{s}: duplicate uid"

# keyboard residue on test: the one target with keyboard, a musical cue and no other device
# keyword. Dropping its 7 rows gives the sensitivity figures in the notes.
kb = [r for r in SPLITS["test"] if r["output_image"] == "triple/target3/11_3686.png"]
assert len(kb) == 7, "keyboard-residue target row count moved"

print(f"screen 302/62 nested in train, exclusion leaves {kept} rows, uids unique, keyboard residue pinned: PASS")

screen 302/62 nested in train, exclusion leaves 39591 rows, uids unique, keyboard residue pinned: PASS
